In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as mlp
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import re

mlp.rcParams['font.family'] = 'serif'
mlp.rcParams['font.serif'] = ['Times New Roman'] + mlp.rcParams['font.serif']
mlp.rcParams['pdf.fonttype'] = 42
mlp.rcParams['hatch.linewidth'] = 0.6

APATH="/home/david/g5/tmp/gem5-phast/gem5-svr-bench/evaluation"


def name(cfg):
    n = "Scale "
    match = re.search(r'_f(\d+)_', cfg)
    s =  match.group(1) if match else ""
    n += s + "x"
    if "_infp1_" in cfg:
        n += "\n+ Inf Pred"
    return n

def fmtBytes(size):
    # 2**10 = 1024
    tmp = -size if size < 0 else size
    power = 2**10
    n = 0
    power_labels = {0 : '', 1: 'K', 2: 'M', 3: 'G', 4: 'T'}
    while tmp >= power:
        tmp /= power
        n += 1
    return "%s%d%s" % ("-" if size < 0 else "", tmp, power_labels[n])


def geo_mean(iterable):
    a = np.array(iterable)
    return np.prod(a, where=a>0)**(1.0/len(a))

def fmtByte(value, unit="B"):
    for unit in ["", "k", "M", "G", "T", "P"]:
        if value < 1024:
            return f"{value:.0f}{unit}"
        value /= 1024

def renameBms(bm):
    m = {
        "nodeapp": "NodeApp",
        "mediawiki": "PHPWiki",
        "compression": "Compression",
        "proto": "Proto",

        "swissmap": "SwissMap",
        "tcmalloc": "TCMalloc",
        "stl": "STL",

        "dacapo-h2": "H2",
        "dacapo-h2o": "H2O",
        "dacapo-kafka": "Kafka",
        "dacapo-luindex": "Luindex",
        "dacapo-spring": "Spring",
        "dacapo-tomcat": "Tomcat",
        "dacapo-lusearch": "Lusearch",
        "benchbase-otmetrics": "OTMetrics",
        "benchbase-voter": "Voter",
        "benchbase-twitter": "Twitter",
        "benchbase-tatp": "TATP",
        "benchbase-resourcestresser": "Stresser",
        "benchbase-epinions": "Epinions",
        "benchbase-ycsb": "YCSB",
        "benchbase-seats": "Seats",
        "benchbase-sibench": "SiBench",
        "benchbase-noop": "Noop",
        "benchbase-smallbank": "SmallBank",
        "renaissance-http": "HTTP",
        "renaissance-chirper": "Chirper",


        "500.perlbench_r.inp": "500.perlbench",
        "500.perlbench_r.checkspam": "500.perlbench",
        "500.perlbench_r.diffmail": "500.perlbench",
        "502.gcc_r.gcc-pp.opts-O3_-finline-limit_36000": "502.gcc",
        "502.gcc_r.inp": "502.gcc",

        "505.mcf_r.inp": "505.mcf",

        "520.omnetpp_r.inp": "520.omnetpp",
        "520.omnetpp_r.general": "520.omnetpp",
        "523.xalancbmk_r.inp": "523.xalancbmk",
        "523.xalancbmk_r.xalanc": "523.xalancbmk",
        "525.x264_r.inp": "525.x264",
        "525.x264_r.x264": "525.x264",
        "531.deepsjeng_r.ref": "531.deepsjeng",
        "538.imagick_r.inp": "538.imagick",
        "541.leela_r.inp": "541.leela",
        "541.leela_r.ref": "541.leela",
        "544.nab_r.inp": "544.nab",
        "557.xz_r.input": "557.xz",
        "557.xz_r.cld": "557.xz",
        "557.xz_r.cpu2006docs": "557.xz",

        ## FP
        "508.namd_r.apoa1": "508.namd",
        "510.parest_r.ref": "510.parest",
        "511.povray_r.ref": "511.povray",
        "519.lbm_r.ref": "519.lbm",
        "526.blender_r.ref": "526.blender",
        "538.imagick_r.ref": "538.imagick",
        "544.nab_r.ref": "544.nab",


    }
    return m.get(bm, bm) if bm else bm


small_bms = [
    "nodeapp",
    "mediawiki",
    "compression",
    "dacapo-spring",
    "dacapo-luindex",
    "dacapo-lusearch",
    "renaissance-http",
    "renaissance-chirper",

    "502.gcc_r.gcc-pp.opts-O3_-finline-limit_36000",
    "505.mcf_r.inp",
    "523.xalancbmk_r.xalanc",
    "531.deepsjeng_r.ref",
    "541.leela_r.ref",
    "557.xz_r.cpu2006docs",

]



all_bms = [
    "nodeapp",
    "mediawiki",

    "proto",
    # "swissmap",
    "tcmalloc",
    "stl",

    "dacapo-h2",
    "dacapo-h2o",
    # "dacapo-kafka",
    "dacapo-luindex",
    "dacapo-spring",
    "dacapo-tomcat",
    "dacapo-lusearch",


    # "benchbase-otmetrics",
    "benchbase-voter",
    "benchbase-twitter",
    "benchbase-tatp",
    # "benchbase-resourcestresser",
    "benchbase-epinions",
    "benchbase-ycsb",
    "benchbase-seats",
    "benchbase-sibench",
    "benchbase-noop",
    "benchbase-smallbank",


    "dacapo-spring",
    "dacapo-luindex",
    "dacapo-lusearch",
    "renaissance-http",
    "renaissance-chirper",

    "renaissance-http",

    "500.perlbench_r.checkspam",
    "502.gcc_r.gcc-pp.opts-O3_-finline-limit_36000",
    "505.mcf_r.inp",
    "508.namd_r.apoa1",
    "510.parest_r.ref",
    "511.povray_r.ref",
    "519.lbm_r.ref",
    "520.omnetpp_r.general",
    "523.xalancbmk_r.xalanc",
    "525.x264_r.x264",
    "526.blender_r.ref",
    "531.deepsjeng_r.ref",
    "538.imagick_r.ref",
    "541.leela_r.ref",
    "544.nab_r.ref",

    # "500.perlbench_r.diffmail",
    # "557.xz_r.cpu2006docs",
    # "557.xz_r.cld",
    "557.xz_r.input",


]


# Different parameters

In [ ]:

stats_keys = {
    "{core}.commitStats0.numInsts" : "insts",
    "{core}.numCycles" : "cycles",
    "{core}.fetchStats0.fetchRate" : "fetchRate",

    "{core}.branchPred.mispredictDueToBTBMiss_0::total": "BTB Misses",
    "{core}.branchPred.mispredicted_0::total" : "BPU Misses",
    "{core}.branchPred.mispredictDueToPredictor_0::DirectCond" : "CBP Misses",




    "board.cache_hierarchy.l1icaches{nc}.demandMshrMisses::total" : "L1I MSHR Misses",
    "board.cache_hierarchy.l1icaches{nc}.demandMisses::total" :  "L1I Misses",
    "board.cache_hierarchy.l1dcaches{nc}.demandMshrMisses::total" : "L1D MSHR Misses",
    "board.cache_hierarchy.l1icaches{nc}.demandAvgMissLatency::total" : "L1I MissLat",

    "board.cache_hierarchy.l2caches{nc}.demandMshrMisses::processor.cores{nc}.core.inst" : "L2I MSHR Misses",
    "board.cache_hierarchy.l2caches{nc}.demandMisses::processor.cores{nc}.core.inst" : "L2I Misses",
    "board.cache_hierarchy.l2caches{nc}.demandMshrMisses::processor.cores{nc}.core.data" : "L2D MSHR Misses",

    "board.cache_hierarchy.l3cache.demandMshrMisses::processor.cores{nc}.core.inst" : "L3I MSHR Misses",
    "board.cache_hierarchy.l3cache.demandMshrMisses::processor.cores{nc}.core.data" : "L3D MSHR Misses",


    "board.cache_hierarchy.l3cache.demandMshrMisses::processor.cores{nc}.core.data" : "L3D MSHR Misses",

}


mpkis =  [ k.strip(" Misses") for k in stats_keys.values() if "Misses" in k]


bms = all_bms


# bms = [
#     "nodeapp",
#     # "mediawiki",

#     # "500.perlbench_r.checkspam",
#     "502.gcc_r.gcc-pp.opts-O3_-finline-limit_36000",

#     # "520.omnetpp_r.general",
#     # "523.xalancbmk_r.xalanc",


# ]



configs = [

    "baseline-noFDP",
    "baseline",
    "baseline-bpuLat",
#   "baseline-fdp-bpuLat",
    "twoTaken",
    "twoTaken-parallelfetch",

]



metrics = [
    "insts",
    # # # "IPC",
    "Speedup",
    "fetchRate",
    "BPU MPKI",
    "BTB MPKI",

    # "Bimodal MPKI",
#     "L2I MSHR MPKI",

#     "L1I MSHR MPKI",
#     "L1I MPKI",
]




RES_DIR = "/home/david/g5/tmp/gem5-phast/gem5-svr-bench/results"

data = {}

for cfg in configs:
    data[cfg] = {}
    for bm in bms:
        print(cfg, bm)
        path = f"{RES_DIR}/{cfg}/{bm}/stats.txt"
        with open(path, "r") as f:
            stats = {}
            for line in f:
                try:
                    parts = line.split()
                    stat_name = parts[0]
                    stat_value = parts[1]
                    stats[stat_name] = float(stat_value)
                except:
                    continue
            data[cfg][bm] = stats

        ## Map metrics
        for metric in stats_keys.keys():
            spec = bm.startswith("5")
            mm = metric.format(
                core="board.processor.cores.core" if spec else "board.processor.cores1.core",
                nc="" if spec else "1"
            )
            if mm not in data[cfg][bm]:
                print(f"Warning: Metric {mm} not found in {cfg}/{bm}")
            else:
                data[cfg][bm][stats_keys[metric]] = data[cfg][bm][mm]

        ## Compute derived metrics
        data[cfg][bm]["IPC"] = data[cfg][bm]["insts"] / data[cfg][bm]["cycles"]
        data[cfg][bm]["CPI"] = data[cfg][bm]["cycles"] / data[cfg][bm]["insts"]
        for mpki in mpkis:
            data[cfg][bm][f"{mpki} MPKI"] = 1000 * data[cfg][bm][f"{mpki} Misses"] / data[cfg][bm]["insts"]

        cfgref = configs[0]
        data[cfg][bm]["Speedup"] = data[cfg][bm]["IPC"] / data[cfgref][bm]["IPC"]



bms_ = sorted(bms)


rename = {
    "baseline-fdp" : "Baseline",
}


sc=25
fig, axs = mlp.subplots(len(metrics), 1, figsize=(sc, sc*len(metrics)*0.3), sharex=True)
# fig, axs = mlp.subplots(1, 1, figsize=(sc, sc*len(metrics)*0.4), sharex=True)

# axs = [axs]
if len(metrics) ==1:
    axs = [axs]

width = 0.9/len(configs)  # the width of the bars

plot_mean = True

x = np.arange(len(bms_))  # the label locations
if plot_mean:
    x = x.tolist() + [len(bms_) + 0.3]


for i, metric in enumerate(metrics):
    ax = axs[i]
    # ax=axs[0]

    for j, cfg in enumerate(configs):
        y = [data[cfg][bm][metric] for bm in bms_]
        if plot_mean:
            y += [np.mean(y)]
        l = rename[cfg] if cfg in rename else cfg
        ax.bar(x=np.array(x) + j*width - (len(configs)-1)*width/2, height=y, width=width, label=l)



    # Add some text for labels, title and custom x-axis tick labels, etc.
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.set_xticks(x)
    ax.grid(axis='y')

    # ax.set_ylim(top=260)

    _bms = bms_ + (["Mean"] if plot_mean else [])
    ax.set_xticklabels(_bms, rotation=45, ha='right')


    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))

    if metric == "Speedup":
        ax.set_ylim(bottom=0.99)


for ax in axs:
    for container in ax.containers:
        ax.bar_label(container, fmt='%.4f', label_type="edge", fontsize=8, rotation=90, zorder=3)
